All the Necessary packages can be installed in this first Section

In [1]:
%pip install transformers datasets evaluate scikit-learn rouge_score bert_score
%pip install -q --upgrade torchao

from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                         TrainingArguments, Trainer, DataCollatorWithPadding)
import torch
import numpy as np
from datasets import load_dataset
import evaluate, time
# Import performance metrics used for Evaluation
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
import nltk


Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=25027 sha256=8cb375f378cc841dde5d8123cadabd383324e3fc28dd3cc7d7aadcc57de2030d
  Stored in directory: c:\users\ruben\appdata\local\packages\pythonsoftwarefoundation.python.3.12_qbz5n2kfra8p0\localcache\local\pip\cache\wheels\85\9d\af\01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score

   ---------------------------------------- 0/4 [absl-py]
   ---------- ----------------------------- 1/4 [rouge_score]
   -------------------- ------------------- 2/4 [evaluate]
   ------------

W0525 18:18:56.709000 28408 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading the Dataset, Splitting dataset into Train and Test set

In [ ]:
file_path = ""

dataset = load_dataset("csv", data_files=file_path)

dataset = dataset["train"].rename_column("MGT", "labels")

split_dataset = dataset.train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

Intialize Model, Prepare setup for training

In [ ]:
# model_id = "Davlan/afro-xlmr-large" # This is the AfroXLMR-large model
# model_id = "Jacaranda/Xhosa_ZuluLlama3_v1" # Xhosa_ZuluLlama3 Model
model_id = "FacebookAI/xlm-roberta-base" # XLM-RoBERTa-base Model


tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess(examples):
  return tokenizer(examples['text'], truncation=False, padding="longest")

tokenized_train = train_dataset.map(preprocess, batched=True)
tokenized_test = test_dataset.map(preprocess, batched=True)

tokenized_train = tokenized_train.remove_columns(["id", "source", "text"])
tokenized_test  = tokenized_test.remove_columns(["id", "source", "text"])

# Intialize the Specific Model using Model_id. 2 Labels for MGT/Not MGT
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels = 2)

print(f"Total parameters to train: {sum(p.numel() for p in model.parameters()):,}")

Trainging and Evaluation Setups

In [ ]:
# Load all classification metrics
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy  = accuracy_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels, average="weighted")
    recall    = recall_metric.compute(predictions=predictions, references=labels, average="weighted")
    f1        = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    return {
        "accuracy":  accuracy["accuracy"],
        "precision": precision["precision"],
        "recall":    recall["recall"],
        "f1":        f1["f1"],
    }



# Setup Training Arguments

full_train_args = TrainingArguments(
    output_dir="./full_results",          # where to save model checkpoints
    num_train_epochs=3,                   # 3-5 is typical since model is pretrained. Increase/decrease it if you want to test
    per_device_train_batch_size=16,       # increase/decrease as your GPU memory allows
    per_device_eval_batch_size=16,        # batch size for evaluation
    eval_strategy="epoch",                # evaluate at end of each epoch
    save_strategy="epoch",                # save checkpoint each epoch
    load_best_model_at_end=True,          # load best checkpoint when done
    metric_for_best_model="accuracy",     # use F1 or Accuracy to determine best model. Balanced set means Accuracy is viable
    weight_decay=0.01,                    # regularization to prevent overfitting
    warmup_ratio=0.1,                     # warm up LR for first 10% of steps
    logging_steps=10,                     
    fp16=True,                            # mixed precision, faster if GPU supports it
    seed=42,                              
)


full_trainer = Trainer(
    model = model,
    args=full_train_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

For BLEU, METEOR, ROUGE-L and BERTScore Calculation Setup

In [ ]:
nltk.download("wordnet")

# generated_text - Text that was generated with other LLM's. MGT
# reference_texts = Text that was gathered from datasets. Not MGT (HGT)
def compute_similarity_metrics(generated_texts, reference_texts):
    # BLEU
    references_tokenized  = [[ref.split()] for ref in reference_texts]
    hypotheses_tokenized  = [gen.split() for gen in generated_texts]
    bleu = corpus_bleu(references_tokenized, hypotheses_tokenized)

    # METEOR
    meteor_scores = [
        meteor_score([ref.split()], gen.split())
        for ref, gen in zip(reference_texts, generated_texts)
    ]
    avg_meteor = np.mean(meteor_scores)

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [
        scorer.score(ref, gen)["rougeL"].fmeasure
        for ref, gen in zip(reference_texts, generated_texts)
    ]
    avg_rouge = np.mean(rouge_scores)

    # BERTScore
    P, R, F1 = bert_score(generated_texts, reference_texts, lang="en")
    avg_bertscore = F1.mean().item()

    return {
        "BLEU":      bleu,
        "METEOR":    avg_meteor,
        "ROUGE-L":   avg_rouge,
        "BERTScore": avg_bertscore,
    }

Train and Evaluate the Model and Calculate Metrics

In [ ]:
# Timer to Track Duration
start_time = time.time()

# Train Model
full_trainer.train()

# Evaluate classification metrics
full_eval_results = full_trainer.evaluate()
full_duration = time.time() - start_time
print(f"Full Fine-tuning took: {full_duration:.2f} seconds")
print(f"Classification Results: {full_eval_results}")

# Get predictions on test set
predictions_output = full_trainer.predict(tokenized_test)
logits = predictions_output.predictions
predicted_labels = np.argmax(logits, axis=-1)
print(f"Predicted Labels Distribution: {np.bincount(predicted_labels)}")


# Separate texts by class for similarity comparison
human_texts = [text for text, label in zip(test_dataset["text"], test_dataset["labels"]) if label == 0]
mgt_texts   = [text for text, label in zip(test_dataset["text"], test_dataset["labels"]) if label == 1]

# Compute similarity metrics (MGT vs Human text)
similarity_results = compute_similarity_metrics(mgt_texts, human_texts)
print(f"Similarity Results: {similarity_results}")